# Bybit Copula Framework - Complete Tutorial

This notebook demonstrates the complete Bybit Copula Framework, including:
1. Data fetching from Bybit
2. Copula fitting and analysis
3. Trading strategy development
4. Backtesting

## Table of Contents
1. [Introduction to Copulas](#intro)
2. [Data Fetching](#data)
3. [Copula Fitting](#fitting)
4. [Dependence Analysis](#analysis)
5. [Trading Strategies](#strategies)
6. [Backtesting](#backtesting)

In [ ]:
# Import required libraries
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.bybit_fetcher import BybitDataFetcher
from src.data.returns import ReturnCalculator
from src.manager.copula_manager import CopulaManager
from src.analysis.dependence import DependenceMetrics
from src.analysis.risk import RiskMetrics

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")

## 1. Introduction to Copulas <a name="intro"></a>

### What are Copulas?

Copulas are functions that link univariate marginal distributions to form multivariate distributions. They separate the dependence structure from the marginal distributions.

**Key Formula:**
$$F(x,y) = C(F_X(x), F_Y(y))$$

Where:
- $F(x,y)$ is the joint distribution
- $F_X(x), F_Y(y)$ are marginal CDFs
- $C$ is the copula function

### Available Copulas

1. **Gaussian Copula**: Symmetric, no tail dependence
2. **Student-t Copula**: Heavy tails, symmetric tail dependence
3. **Clayton Copula**: Lower tail dependence
4. **Gumbel Copula**: Upper tail dependence
5. **Frank Copula**: Symmetric, no tail dependence

## 2. Data Fetching <a name="data"></a>

Let's fetch historical 15-minute data from Bybit for BTC and ETH.

In [ ]:
# Initialize data fetcher
fetcher = BybitDataFetcher()

# Fetch data
symbol1 = "BTCUSDT"
symbol2 = "ETHUSDT"
interval = "15"  # 15-minute bars
days = 30

print(f"Fetching {days} days of {interval}-minute data...")

df1 = fetcher.fetch_and_cache(symbol1, interval=interval, days=days)
df2 = fetcher.fetch_and_cache(symbol2, interval=interval, days=days)

print(f"\n{symbol1}: {len(df1)} bars")
print(f"{symbol2}: {len(df2)} bars")

# Display first few rows
df1.head()

In [ ]:
# Plot price series
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df1['datetime'], df1['close'], label=symbol1, linewidth=1.5)
axes[0].set_title(f'{symbol1} Price', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price (USD)', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df2['datetime'], df2['close'], label=symbol2, color='orange', linewidth=1.5)
axes[1].set_title(f'{symbol2} Price', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Price (USD)', fontsize=12)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Copula Fitting <a name="fitting"></a>

Now let's fit multiple copula models and compare them.

In [ ]:
# Initialize copula manager
manager = CopulaManager()

# Fit copulas
print("Fitting copula models...\n")

results = manager.fit_pair(
    symbol1=symbol1,
    symbol2=symbol2,
    interval=interval,
    days=days,
    copula_types=['gaussian', 'student_t', 'clayton', 'gumbel', 'frank']
)

print("\n" + "="*80)
print("Copula Comparison")
print("="*80)

In [ ]:
# Create comparison DataFrame
comparison_data = []

for copula_type, result in results['copulas'].items():
    comparison_data.append({
        'Copula': copula_type,
        'AIC': result['aic'],
        'BIC': result['bic'],
        'Log-Likelihood': result['log_likelihood'],
        "Kendall's τ": result['kendall_tau'],
        'Lower Tail λ': result['tail_dependence']['lower'],
        'Upper Tail λ': result['tail_dependence']['upper'],
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('AIC')

print(comparison_df.to_string(index=False))

# Highlight best model
best = results['best_copula']
print(f"\n✓ Best model: {best['type']} (AIC={best['aic']:.2f})")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AIC comparison
axes[0].barh(comparison_df['Copula'], comparison_df['AIC'])
axes[0].set_xlabel('AIC (lower is better)', fontsize=12)
axes[0].set_title('Model Selection: AIC', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Tail dependence comparison
x = np.arange(len(comparison_df))
width = 0.35

axes[1].bar(x - width/2, comparison_df['Lower Tail λ'], width, label='Lower Tail', alpha=0.8)
axes[1].bar(x + width/2, comparison_df['Upper Tail λ'], width, label='Upper Tail', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(comparison_df['Copula'], rotation=45)
axes[1].set_ylabel('Tail Dependence λ', fontsize=12)
axes[1].set_title('Tail Dependence Coefficients', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4. Dependence Analysis <a name="analysis"></a>

Let's analyze the dependence structure in detail.

In [ ]:
# Get returns
r1 = results['data']['returns1']
r2 = results['data']['returns2']

# Compute all dependence metrics
metrics = DependenceMetrics.all_metrics(r1, r2)

print("Dependence Metrics:")
print(f"  Pearson correlation:  {metrics['pearson']:.4f}")
print(f"  Spearman's rho:       {metrics['spearman']:.4f}")
print(f"  Kendall's tau:        {metrics['kendall']:.4f}")
print(f"\nTail Dependence (empirical):")
print(f"  Lower tail:           {metrics['tail_dependence']['lower']:.4f}")
print(f"  Upper tail:           {metrics['tail_dependence']['upper']:.4f}")
print(f"\nCorrelation Breakdown:")
for region, corr in metrics['correlation_breakdown'].items():
    print(f"  {region.capitalize():<10}: {corr:.4f}")

In [ ]:
# Visualize returns and dependence
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Returns scatter plot
axes[0, 0].scatter(r1, r2, alpha=0.5, s=10)
axes[0, 0].set_xlabel(f'{symbol1} Returns', fontsize=12)
axes[0, 0].set_ylabel(f'{symbol2} Returns', fontsize=12)
axes[0, 0].set_title('Returns Scatter Plot', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Returns distribution
axes[0, 1].hist(r1, bins=50, alpha=0.7, label=symbol1, edgecolor='black')
axes[0, 1].hist(r2, bins=50, alpha=0.7, label=symbol2, edgecolor='black')
axes[0, 1].set_xlabel('Returns', fontsize=12)
axes[0, 1].set_ylabel('Frequency', fontsize=12)
axes[0, 1].set_title('Returns Distribution', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Uniform marginals (from copula transformation)
U = results['data']['uniform']
axes[1, 0].scatter(U[:,0], U[:,1], alpha=0.5, s=10, color='green')
axes[1, 0].set_xlabel('U1 (uniform)', fontsize=12)
axes[1, 0].set_ylabel('U2 (uniform)', fontsize=12)
axes[1, 0].set_title('Copula Space (Uniform Marginals)', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Rolling correlation
rolling_corr = pd.Series(r1).rolling(window=100).corr(pd.Series(r2))
axes[1, 1].plot(rolling_corr, linewidth=1.5)
axes[1, 1].axhline(y=metrics['pearson'], color='r', linestyle='--', label='Mean correlation')
axes[1, 1].set_xlabel('Time', fontsize=12)
axes[1, 1].set_ylabel('Correlation', fontsize=12)
axes[1, 1].set_title('Rolling Correlation (100-bar window)', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Trading Strategies <a name="strategies"></a>

Now let's implement a simple pairs trading strategy using the fitted copula.

In [ ]:
from src.strategies.pairs_trading import CopulaPairsTradingStrategy

# Create strategy
strategy = CopulaPairsTradingStrategy(
    symbol1=symbol1,
    symbol2=symbol2,
    lookback_period=500,
    copula_type='gaussian',
    entry_threshold=0.05,
    exit_threshold=0.5,
    refit_frequency=100,
    position_size=0.5
)

print("Strategy Configuration:")
print(f"  Type: Copula Pairs Trading")
print(f"  Pair: {symbol1} - {symbol2}")
print(f"  Copula: {strategy.copula_type}")
print(f"  Entry threshold: {strategy.entry_threshold}")
print(f"  Position size: {strategy.position_size * 100}%")

## 6. Backtesting <a name="backtesting"></a>

Let's backtest the strategy on historical data.

In [ ]:
from src.backtest.backtest_engine import BacktestEngine

# Prepare data
data = {
    symbol1: df1,
    symbol2: df2
}

# Initialize backtest engine
engine = BacktestEngine(
    initial_capital=100000.0,
    transaction_cost=0.001,  # 0.1%
    slippage=0.0005  # 0.05%
)

# Load data
engine.load_data(data)

# Run backtest
print("Running backtest...")
backtest_results = engine.run(strategy, show_progress=True)

print("\nBacktest complete!")

In [ ]:
# Print results
engine.print_results()

In [ ]:
# Plot results
engine.plot_results()

In [ ]:
# Analyze trades
trades_df = backtest_results['trades']

if len(trades_df) > 0:
    print(f"\nTrade Analysis:")
    print(f"  Total trades: {len(trades_df)}")
    
    # Trades by symbol
    print(f"\n  Trades by symbol:")
    print(trades_df['symbol'].value_counts())
    
    # Average trade value
    avg_value = trades_df['value'].mean()
    print(f"\n  Average trade value: ${avg_value:,.2f}")
    
    # Total transaction costs
    total_costs = trades_df['cost'].sum()
    print(f"  Total transaction costs: ${total_costs:,.2f}")
    
    # Display sample trades
    print(f"\nSample trades:")
    print(trades_df[['timestamp', 'symbol', 'quantity', 'price', 'value']].head(10))

## Summary

In this tutorial, we:

1. **Fetched data** from Bybit for BTC and ETH
2. **Fitted multiple copula models** and compared them using AIC/BIC
3. **Analyzed dependencies** using various metrics including tail dependence
4. **Implemented a pairs trading strategy** based on copula conditional probabilities
5. **Backtested the strategy** and analyzed performance

### Key Takeaways

- **Copulas** separate dependence structure from marginal distributions
- **Different copula types** capture different dependency patterns (tail dependence)
- **Model selection** using AIC/BIC helps choose the best-fitting copula
- **Copula-based strategies** can exploit mispriced relationships
- **Backtesting** validates strategy performance on historical data

### Next Steps

Try:
- Different cryptocurrency pairs
- Other copula types
- Alternative strategy parameters
- Statistical arbitrage with multiple pairs
- Tail risk hedging strategies